In [ ]:
"""
NOTE:
The scripts contain local absolute paths used during thesis experiments.
To rerun the code, adapt the path definitions
(e.g., IMG_DIR, COCO_JSON, BASE_OUT_DIR) to match your local directory structure.
"""

# -------------
# Figures 36-37
# -------------

import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import seaborn as sns


# ---------------- CONFIG ---------------- #
PYRO_DIR = "path/to/pyro-sdis/images"
FIGLIB_DIR = "path/to/figlib"  # contains event subfolders
SAVE_DIR = "path/to/save/dir"


# Academic matplotlib defaults
plt.rcParams.update(
    {
        "figure.dpi": 300,
        "font.family": "serif",
        "font.size": 11,
        "axes.labelsize": 12,
        "axes.titlesize": 13,
        "legend.fontsize": 10,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "axes.grid": True,
        "grid.linestyle": "--",
        "grid.alpha": 0.6,
    }
)

os.makedirs(SAVE_DIR, exist_ok=True)

# ================================================================
# FUNCTION: Compute brightness & contrast
# ================================================================
def compute_brightness_contrast(image_paths: list[Path]) -> tuple[np.ndarray, np.ndarray]:
    """Compute brightness (mean) and contrast (std) for image collection."""
    brightness_list = []
    contrast_list = []

    for path in tqdm(image_paths, desc=f"Processing {len(image_paths)} images"):
        try:
            img = Image.open(path).convert("RGB")
            arr = np.array(img) / 255.0

            brightness_list.append(arr.mean())
            contrast_list.append(arr.std())

        except Exception:
            continue

    return np.array(brightness_list), np.array(contrast_list)


# ================================================================
# FUNCTION: Collect images recursively (FIgLib) or flat (Pyro)
# ================================================================
def collect_images(folder: str | Path, recursive: bool = False) -> list[Path]:
    """Collect all image files from folder."""
    folder = Path(folder)
    if recursive:
        return (
            list(folder.rglob("*.jpg")) + list(folder.rglob("*.png"))
        )
    return list(folder.glob("*.jpg")) + list(folder.glob("*.png"))


# ==================
# PLOT FUNCTIONS
# ==================
def plot_hist_kde(
    pyro_vals: np.ndarray,
    figlib_vals: np.ndarray,
    xlabel: str,
    title: str,
    save_name: str,
) -> None:
    """Histogram + KDE comparison."""
    plt.figure(figsize=(8, 4))

    sns.histplot(pyro_vals, bins=40, kde=True, color="#1e90ff", alpha=0.5, label="Pyro-SDIS")
    sns.histplot(figlib_vals, bins=40, kde=True, color="#daa520", alpha=0.5, label="FIgLib")

    plt.xlabel(xlabel)
    plt.ylabel("Count")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(Path(SAVE_DIR) / save_name, dpi=300)
    plt.close()


def plot_scatter(
    pyro_bright: np.ndarray,
    pyro_contrast: np.ndarray,
    figlib_bright: np.ndarray,
    figlib_contrast: np.ndarray,
    save_name: str,
) -> None:
    """Brightness vs contrast scatter plot."""
    plt.figure(figsize=(8, 4))

    plt.scatter(
        pyro_bright,
        pyro_contrast,
        s=10,
        alpha=0.5,
        color="#1e90ff",
        label="Pyro-SDIS",
    )

    plt.scatter(
        figlib_bright,
        figlib_contrast,
        s=10,
        alpha=0.5,
        color="#daa520",
        label="FIgLib",
    )

    plt.xlabel("Brightness (mean)")
    plt.ylabel("Contrast (std)")
    plt.title("Brightness–Contrast Relationship")
    plt.legend()
    plt.tight_layout()
    plt.savefig(Path(SAVE_DIR) / save_name, dpi=300)
    plt.close()


def plot_2d_kde(
    pyro_bright: np.ndarray,
    pyro_contrast: np.ndarray,
    figlib_bright: np.ndarray,
    figlib_contrast: np.ndarray,
    save_name: str,
) -> None:
    """2D KDE density plot."""
    plt.figure(figsize=(7, 6))

    # Pyro-SDIS
    sns.kdeplot(
        x=pyro_bright,
        y=pyro_contrast,
        cmap="Blues",
        fill=True,
        thresh=0.02,
        alpha=0.5,
        label="Pyro-SDIS",
    )

    # FIgLib
    sns.kdeplot(
        x=figlib_bright,
        y=figlib_contrast,
        cmap="Oranges",
        fill=True,
        thresh=0.02,
        alpha=0.5,
        label="FIgLib",
    )

    plt.xlabel("Brightness (mean)")
    plt.ylabel("Contrast (std)")
    plt.title("2D KDE Density: Brightness × Contrast")
    plt.legend()
    plt.tight_layout()
    plt.savefig(Path(SAVE_DIR) / save_name, dpi=300)
    plt.close()


def plot_violin(
    pyro_vals: np.ndarray,
    figlib_vals: np.ndarray,
    xlabel: str,
    title: str,
    save_name: str,
) -> None:
    """Violin plot comparison."""
    plt.figure(figsize=(8, 4))

    df = pd.DataFrame(
        {
            "Value": np.concatenate([pyro_vals, figlib_vals]),
            "Dataset": (
                ["Pyro-SDIS"] * len(pyro_vals)
                + ["FIgLib"] * len(figlib_vals)
            ),
        }
    )

    sns.violinplot(data=df, x="Dataset", y="Value", palette=["#1e90ff", "#daa520"])

    plt.xlabel("Dataset")
    plt.ylabel(xlabel)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(Path(SAVE_DIR) / save_name, dpi=300)
    plt.close()


# ========
# MAIN
# ========
if __name__ == "__main__":

    print("\n=============== LOADING PYRO-SDIS ===============")
    pyro_images = collect_images(PYRO_DIR, recursive=False)
    pyro_brightness, pyro_contrast = compute_brightness_contrast(pyro_images)

    print("\n=============== LOADING FIgLib ===============")
    figlib_images = collect_images(FIGLIB_DIR, recursive=True)
    figlib_brightness, figlib_contrast = compute_brightness_contrast(figlib_images)

    print("\n=============== SAVING PLOTS ===============")

    # 1) Brightness Distribution
    plot_hist_kde(
        pyro_brightness,
        figlib_brightness,
        xlabel="Brightness",
        title="Brightness Distribution Across Pyro-SDIS and FIgLib",
        save_name="brightness_comparison.png",
    )

    # 2) Contrast Distribution
    plot_hist_kde(
        pyro_contrast,
        figlib_contrast,
        xlabel="Contrast",
        title="Contrast Distribution Across Pyro-SDIS and FIgLib",
        save_name="contrast_comparison.png",
    )

    # 3) Scatter Plot
    plot_scatter(
        pyro_brightness,
        pyro_contrast,
        figlib_brightness,
        figlib_contrast,
        save_name="brightness_contrast_scatter.png",
    )

    # 4) 2D KDE
    plot_2d_kde(
        pyro_brightness,
        pyro_contrast,
        figlib_brightness,
        figlib_contrast,
        save_name="brightness_contrast_kde2d.png",
    )

    # 5) Violin Plot
    plot_violin(
        pyro_brightness,
        figlib_brightness,
        xlabel="Brightness",
        title="Brightness Variation Across Datasets",
        save_name="brightness_violin.png",
    )

    plot_violin(
        pyro_contrast,
        figlib_contrast,
        xlabel="Contrast",
        title="Contrast Variation Across Datasets",
        save_name="contrast_violin.png",
    )

    print("\nSaved all plots in:", SAVE_DIR)
